# DLAI Model Merging - Specialist pilot (seed 42)

This is the first meaningful training run. Every task receives exactly 400 optimizer steps so that task-vector comparisons are not confounded by different dataset sizes. The run uses one seed; final evidence will use multiple seeds only after this pilot is validated.

Kaggle settings: **GPU T4 x2** (not P100), Internet on. The code uses one T4; the second is not required.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = 'https://github.com/LeuxLello/Dlai-model-merging.git'
WORKDIR = Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(WORKDIR)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(WORKDIR)])
sys.path.insert(0, str(WORKDIR / 'src'))
os.chdir(WORKDIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import json, platform, torch, transformers, datasets
from dlai_merge.training import TrainConfig, train_specialist

assert torch.cuda.is_available(), 'Enable a Kaggle GPU before running.'
gpu_name = torch.cuda.get_device_name(0)
compute_capability = torch.cuda.get_device_capability(0)
print('GPU:', gpu_name, '| compute capability:', compute_capability)
assert compute_capability[0] >= 7, (
    f'GPU {gpu_name} with capability {compute_capability} is incompatible with this Kaggle PyTorch build. '
    'Stop the session and select GPU T4 x2 in Notebook options.'
)
torch.ones(1, device='cuda').add_(1)  # fail fast if CUDA kernels are incompatible
environment = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'datasets': datasets.__version__,
    'gpu': gpu_name,
    'compute_capability': list(compute_capability),
}
environment

## Train four specialists
Large datasets are capped at 12,000 deterministic training examples and 2,000 evaluation examples. Small GLUE tasks use all available examples and repeat them as needed to reach the same 400-step optimization budget.

In [ ]:
OUTPUT_ROOT = '/kaggle/working/pilot_specialists'
SEED = 42
summaries = {}

for task in ['sst2', 'imdb', 'mrpc', 'rte']:
    print(f'\n===== TRAINING {task.upper()} =====')
    config = TrainConfig(
        task=task,
        output_root=OUTPUT_ROOT,
        seed=SEED,
        max_train_samples=12_000,
        max_eval_samples=2_000,
        max_steps=400,
        eval_steps=100,
        train_batch_size=32,
        eval_batch_size=64,
        learning_rate=2e-5,
    )
    summaries[task] = train_specialist(config)
    metric = summaries[task]['primary_metric']
    print(task, metric, summaries[task]['eval_metrics'][f'eval_{metric}'])

## Compact quality table

In [ ]:
import pandas as pd

rows = []
for task, summary in summaries.items():
    rows.append({
        'task': task,
        'primary_metric': summary['primary_metric'],
        'accuracy': summary['eval_metrics']['eval_accuracy'],
        'f1': summary['eval_metrics']['eval_f1'],
        'eval_loss': summary['eval_metrics']['eval_loss'],
        'train_runtime_s': summary['train_metrics']['train_runtime'],
        'train_samples': summary['train_samples'],
        'eval_samples': summary['eval_samples'],
    })
quality = pd.DataFrame(rows)
quality

## Save reproducible outputs
The archive contains the four encoder states, four task heads, tokenizers, and JSON summaries. Trainer-internal checkpoints are removed because the selected final states have already been exported.

In [ ]:
from pathlib import Path

for trainer_dir in Path(OUTPUT_ROOT).glob('*/seed-*/trainer'):
    shutil.rmtree(trainer_dir)
bundle_summary = {
    'purpose': 'specialist pilot; one seed; not final multi-seed evidence',
    'environment': environment,
    'seed': SEED,
    'summaries': summaries,
}
Path('/kaggle/working/pilot_summary.json').write_text(
    json.dumps(bundle_summary, indent=2), encoding='utf-8'
)
quality.to_csv('/kaggle/working/pilot_metrics.csv', index=False)
shutil.make_archive('/kaggle/working/pilot_specialists_seed42', 'zip', OUTPUT_ROOT)
print('/kaggle/working/pilot_summary.json')
print('/kaggle/working/pilot_metrics.csv')
print('/kaggle/working/pilot_specialists_seed42.zip')